In [ ]:
'''
“Linear Regression from Scratch” - Purpose: the first project where you
build the algorithm, not just the pipeline around it — this is what makes gradient
descent click permanently instead of staying an abstract diagram. - Spec:
numpy only, no scikit-learn. Generate or download a small single-feature or
two-feature dataset (e.g., a simple synthetic linear relationship with noise, or a
small real dataset like height-vs-weight). - Deliverables: implement the cost
function (MSE), the gradient computation, and the update loop by hand; plot
the cost decreasing over iterations to confirm convergence; plot your fitted line
against the data; compare your learned coefficients against np.linalg.lstsq
or scikit-learn’s LinearRegression on the same data and report how close they
are. - Stretch goal: implement logistic regression from scratch the same way
(sigmoid + binary cross-entropy loss instead of MSE) on a small 2-class synthetic
dataset — this directly previews Project 1’s core model family.
'''

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('Social_Network_Ads.csv')
df.head(5)

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0


In [3]:
x_train = df['Age'].to_numpy()
x_train

array([19, 35, 26, 27, 19, 27, 27, 32, 25, 35, 26, 26, 20, 32, 18, 29, 47,
       45, 46, 48, 45, 47, 48, 45, 46, 47, 49, 47, 29, 31, 31, 27, 21, 28,
       27, 35, 33, 30, 26, 27, 27, 33, 35, 30, 28, 23, 25, 27, 30, 31, 24,
       18, 29, 35, 27, 24, 23, 28, 22, 32, 27, 25, 23, 32, 59, 24, 24, 23,
       22, 31, 25, 24, 20, 33, 32, 34, 18, 22, 28, 26, 30, 39, 20, 35, 30,
       31, 24, 28, 26, 35, 22, 30, 26, 29, 29, 35, 35, 28, 35, 28, 27, 28,
       32, 33, 19, 21, 26, 27, 26, 38, 39, 37, 38, 37, 42, 40, 35, 36, 40,
       41, 36, 37, 40, 35, 41, 39, 42, 26, 30, 26, 31, 33, 30, 21, 28, 23,
       20, 30, 28, 19, 19, 18, 35, 30, 34, 24, 27, 41, 29, 20, 26, 41, 31,
       36, 40, 31, 46, 29, 26, 32, 32, 25, 37, 35, 33, 18, 22, 35, 29, 29,
       21, 34, 26, 34, 34, 23, 35, 25, 24, 31, 26, 31, 32, 33, 33, 31, 20,
       33, 35, 28, 24, 19, 29, 19, 28, 34, 30, 20, 26, 35, 35, 49, 39, 41,
       58, 47, 55, 52, 40, 46, 48, 52, 59, 35, 47, 60, 49, 40, 46, 59, 41,
       35, 37, 60, 35, 37

In [5]:
y_train = df['Purchased'].to_numpy()
y_train

array([0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1,
       0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0,
       1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0,
       1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
       0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1,

In [6]:
def sigmoid(z):
    g = 1/(1+np.exp(-z))
    return g

In [2]:
def cost_func(x,y,w,b):
    m = x.shape[0]
    cost = 0.0
    for i in range(m):
        z_i = np.dot(x[i], w) + b
        f_wb = sigmoid(z_i)
        cost += -y[i]*np.log(f_wb) - (1-y[i])*np.log(1-f_wb)

    cost = cost / m
    return cost
